# CNNs - Practice Exercise

Build and optimize convolutional networks:
1. Experiment with different filter sizes and pooling
2. Visualize learned filters
3. Compare CNN vs Dense on same data
4. Analyze receptive fields and feature maps

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load Fashion-MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

print(f"Data shape: {X_train.shape}")

## Exercise: Optimize CNN Architecture

Test different CNN configurations: varying filter counts, kernel sizes, and pooling strategies.

In [ ]:
# CNN Config A: Fewer filters, larger kernels
cnn_a = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (5, 5), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (5, 5), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# CNN Config B: More filters, smaller kernels
cnn_b = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# CNN Config C: Deep with many layers
cnn_c = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

configs = {'A (Fewer, Large)': cnn_a, 'B (More, Small)': cnn_b, 'C (Deep)': cnn_c}
results = {}

for name, model in configs.items():
    print(f"\nTraining {name}...")
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = model.fit(X_train, y_train, epochs=8, batch_size=128, validation_split=0.2, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    results[name] = {'history': hist, 'test_acc': test_acc, 'params': model.count_params()}
    print(f"{name}: Test Acc = {test_acc:.4f}, Params = {model.count_params():,}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for name in configs.keys():
    hist = results[name]['history']
    axes[0].plot(hist.history['val_accuracy'], label=name, marker='o', markersize=3)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('CNN Architecture Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

names = list(results.keys())
test_accs = [results[n]['test_acc'] for n in names]
axes[1].bar(names, test_accs, alpha=0.7)
axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_ylim([0.90, 0.95])
for i, v in enumerate(test_accs):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=8, fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nQuestion: How does architecture affect accuracy and efficiency?")